# Hybrid QNN Training on CIC IoMT 2024 WiFi MQTT

This notebook trains a Hybrid QNN model on the CIC IoMT 2024 WiFi MQTT dataset.
It performs binary classification (Attack vs Benign).

In [4]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import pennylane as qml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

ModuleNotFoundError: No module named 'matplotlib'

## 1. Configuration & Data Loading

In [ ]:
# Configuration
DATASET_DIR = "Datasets\CIC IoMT 2024 WiFi MQTT"
TRAIN_FILE = "CIC_IoMT_2024_WiFi_MQTT_train.csv"
TEST_FILE = "CIC_IoMT_2024_WiFi_MQTT_test.csv"

N_ROWS = 50000  # Load subset for faster training/testing
BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 0.001
N_QUBITS = 4
N_LAYERS = 2
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# Load Data
train_path = os.path.join(DATASET_DIR, TRAIN_FILE)
print(f"Loading data from {train_path}...")

df = pd.read_csv(train_path, nrows=N_ROWS)
print(f"Loaded {len(df)} samples.")

# Inspect labels
print("Unique labels:", df['label'].unique())

In [ ]:
# Preprocessing
# Binary Classification: Benign_train -> 0, Others -> 1
df['target'] = df['label'].apply(lambda x: 0 if x == 'Benign_train' else 1)

# Drop label column
X = df.drop(columns=['label', 'target']).values
y = df['target'].values

# Scale Features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=SEED)

# Convert to PyTorch Tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# DataLoaders
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")
print(f"Input Features: {X.shape[1]}")

## 2. Hybrid QNN Model

In [ ]:
dev = qml.device("default.qubit", wires=N_QUBITS)

@qml.qnode(dev, interface="torch")
def qnn_layer(inputs, weights):
    for i in range(N_QUBITS):
        qml.RX(inputs[i], wires=i)
    qml.templates.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

class HybridQNN(nn.Module):
    def __init__(self, input_dim, n_qubits, n_layers):
        super().__init__()
        self.n_qubits = n_qubits
        self.projection = nn.Linear(input_dim, n_qubits)
        self.q_weights = nn.Parameter(
            0.01 * torch.randn(n_layers, n_qubits, 3)
        )
        self.fc = nn.Linear(n_qubits, 1)

    def forward(self, x):
        x_proj = torch.tanh(self.projection(x)) * np.pi
        q_out = []
        for sample in x_proj:
            q = qnn_layer(sample, self.q_weights)
            q = torch.stack(q)
            q_out.append(q)
        q_out = torch.stack(q_out).float()
        return self.fc(q_out)

## 3. Training Loop

In [ ]:
input_dim = X.shape[1]
model = HybridQNN(input_dim, N_QUBITS, N_LAYERS)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.BCEWithLogitsLoss()

train_losses = []

print("Starting training...")
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for X_batch, y_batch in loop:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        loop.set_postfix(loss=loss.item())
    
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f"Epoch {epoch+1} Loss: {avg_loss:.4f}")

## 4. Evaluation & Saving

In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        logits = model(X_batch)
        preds = torch.sigmoid(logits) > 0.5
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_batch.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds)
rec = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")

In [ ]:
# Save Model
save_dir = 'trained_model_ciciomt2024'
os.makedirs(save_dir, exist_ok=True)
model_path = os.path.join(save_dir, 'hybrid_qnn_model.pth')
torch.save(model.state_dict(), model_path)
print(f"Model saved to {model_path}")